# Step 2 · Score every AI article for "health & wellness" (Google Colab)

**What this notebook does**
1. Takes the small `nyt_articles_for_scoring.csv` you exported from DataHub (every `GPT Model` article + a random sample of other NYT articles from the same quarters).
2. Downloads the Kaggle **NYT Articles: 2.1M+ (2000–Present)** metadata table (`nyt-metadata.csv`, 4.6 GB, CC0) and **inner-joins** it to your articles on the article URL. That join adds the **headline, abstract, section, news desk and NYT's own subject keywords** that `news_df_sentiment` doesn't have.
3. Runs a **zero-shot NLI classifier** (`MoritzLaurer/deberta-v3-large-zeroshot-v2.0`) on *headline + abstract + lead paragraph* and turns it into a **health score from 0 to 1** (1 = clearly about health & wellness).
4. **Validates** that score against an independent label: NYT's own editorial tags (section = Health/Well, or subject keywords such as *Mental Health and Disorders*). The model never sees those tags.
5. Saves `ai_health_scores.csv`. You upload that to DataHub for the plot.

**Before you start:** `Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save`. Then run the cells top to bottom (`Runtime ▸ Run all` works too). Total time is usually 10–15 minutes, mostly the Kaggle download.

In [ ]:
# 2.1 · Install the two packages Colab doesn't ship with (~20 s)
!pip -q install kagglehub sentencepiece

In [ ]:
# 2.2 · Imports, settings, GPU check
import os, re, ast, glob, json, time
import numpy as np
import pandas as pd
import torch

USE_GPU = torch.cuda.is_available()
DEVICE = 0 if USE_GPU else -1

# Zero-shot NLI model. Large = most accurate; base is ~3x faster and is used automatically without a GPU.
MODEL_NAME = ("MoritzLaurer/deberta-v3-large-zeroshot-v2.0" if USE_GPU
              else "MoritzLaurer/deberta-v3-base-zeroshot-v2.0")

# The four sub-domains of "Health" from the research question. Each gets its own yes/no probability;
# health_score = the highest of the four ("is this article about ANY part of health & wellness?").
HEALTH_LABELS = {
    "medicine":      "medicine, disease, or health care",
    "public_health": "public health",
    "mental_health": "mental health",
    "wellness":      "wellness, fitness, or nutrition",
}
HYPOTHESIS_TEMPLATE = "This news article is about {}."
THRESHOLD = 0.5          # score >= 0.5  ->  counted as a "health & wellness" article

print("GPU:", torch.cuda.get_device_name(0) if USE_GPU else
      "NONE. It will still work, just slower. For speed: Runtime > Change runtime type > T4 GPU, then re-run.")
print("Model:", MODEL_NAME)

In [ ]:
# 2.3 · Upload the file you downloaded from DataHub (nyt_articles_for_scoring.csv)
from google.colab import files

def newest(pattern):
    hits = sorted(glob.glob(pattern), key=os.path.getmtime)
    return hits[-1] if hits else None

EXPORT_PATH = newest("nyt_articles_for_scoring*.csv")
if EXPORT_PATH is None:
    print("A file picker appears below -> choose nyt_articles_for_scoring.csv from your Downloads folder.")
    files.upload()
    EXPORT_PATH = newest("nyt_articles_for_scoring*.csv")
assert EXPORT_PATH, "Upload nyt_articles_for_scoring.csv first (the file exported from DataHub in Step 1)."

arts = pd.read_csv(EXPORT_PATH, dtype=str, keep_default_na=False)
print(f"Loaded {EXPORT_PATH}: {len(arts):,} rows")
print(arts["group"].value_counts().rename({"ai": "AI (GPT Model > 0)", "baseline": "baseline sample"}))

In [ ]:
# 2.4 · Download the Kaggle NYT metadata table (4.6 GB; about 3-6 minutes on Colab)
import kagglehub
KAGGLE_DATASET = "aryansingh0909/nyt-articles-21m-2000-present"

try:
    kaggle_dir = kagglehub.dataset_download(KAGGLE_DATASET)      # public dataset: usually no login needed
except Exception as err:
    print(f"Anonymous download failed ({type(err).__name__}). Paste your Kaggle API token in the box below.")
    print("Get one at kaggle.com > your profile picture > Settings > API > Create New Token.")
    kagglehub.login()
    kaggle_dir = kagglehub.dataset_download(KAGGLE_DATASET)

csvs = glob.glob(os.path.join(kaggle_dir, "**", "*.csv"), recursive=True)
KAGGLE_CSV = max(csvs, key=os.path.getsize)
print(f"{KAGGLE_CSV}  ({os.path.getsize(KAGGLE_CSV) / 1e9:.2f} GB)")
print("Columns:", pd.read_csv(KAGGLE_CSV, nrows=0).columns.tolist())

In [ ]:
# 2.5 · INNER JOIN on the article URL, streaming the 4.6 GB file in chunks so Colab never runs out of RAM
def url_key(s: pd.Series) -> pd.Series:
    # Same article, slightly different URL strings (http vs https, "www.", "?smid=share", trailing "/")
    # would silently fail an exact join, so both sides are normalized the same way first.
    return (s.fillna("").astype(str).str.strip().str.lower()
             .str.replace(r"^https?://", "", regex=True)
             .str.replace(r"^www\.", "", regex=True)
             .str.replace(r"[?#].*$", "", regex=True)
             .str.rstrip("/"))

arts["url_key"] = url_key(arts["web_url"])
wanted = set(arts["url_key"])

header = pd.read_csv(KAGGLE_CSV, nrows=0).columns.tolist()
WANT = ["web_url", "headline", "abstract", "snippet", "lead_paragraph", "keywords",
        "section_name", "subsection_name", "news_desk", "type_of_material", "word_count"]
usecols = [c for c in header if c in WANT or c.startswith("headline.")]

pieces, seen, t0 = [], 0, time.time()
for chunk in pd.read_csv(KAGGLE_CSV, usecols=usecols, dtype=str, keep_default_na=False,
                         chunksize=200_000, on_bad_lines="skip"):
    k = url_key(chunk["web_url"])
    hit = k.isin(wanted)
    if hit.any():
        pieces.append(chunk.loc[hit].assign(url_key=k[hit]))
    seen += len(chunk)
    if (seen // 200_000) % 3 == 0:
        print(f"  scanned {seen:>10,} Kaggle rows | matches so far: {sum(map(len, pieces)):,} | {time.time() - t0:5.0f}s")

kag = (pd.concat(pieces, ignore_index=True)
         .drop_duplicates("url_key", keep="last")          # the daily-updated file can repeat an article
         .drop(columns=["web_url"])                          # keep news_df_sentiment's own URL for the DataHub join
         .rename(columns={"lead_paragraph": "kaggle_lead_paragraph"}))

merged = arts.merge(kag, on="url_key", how="inner")         # <-- the inner join
print(f"\nScanned {seen:,} Kaggle rows in {time.time() - t0:.0f}s")
rate = (merged["group"].value_counts() / arts["group"].value_counts()).fillna(0)
for g in ["ai", "baseline"]:
    print(f"  {g:9s}: {int((merged['group'] == g).sum()):>6,} of {int((arts['group'] == g).sum()):>6,} matched ({rate.get(g, 0):.1%})")

unmatched = arts.loc[~arts["url_key"].isin(merged["url_key"])]
print(f"\n{len(unmatched):,} articles dropped by the inner join. A few examples (worth a look for your limitations section):")
print(unmatched[["group", "web_url"]].head(8).to_string(index=False))

In [ ]:
# 2.6 · Unpack the nested NYT fields and build the text the model reads
def parse_obj(s):
    # Kaggle stores nested API objects as text: "{'main': 'Headline', ...}" or "[{'name': 'subject', 'value': ...}]"
    if not isinstance(s, str) or not s.strip():
        return None
    s = s.strip()
    if s[0] in "[{":
        for loader in (ast.literal_eval, json.loads):
            try:
                return loader(s)
            except Exception:
                pass
    return s

def headline_text(s):
    o = parse_obj(s)
    if isinstance(o, dict):
        return str(o.get("main") or o.get("print_headline") or "").strip()
    if isinstance(o, str):
        m = re.search(r"['\"]main['\"]\s*:\s*(['\"])(.*?)\1", o)
        return m.group(2) if m else o
    return ""

def keyword_list(s):
    o = parse_obj(s)
    if isinstance(o, list):
        return [str(d.get("value")) for d in o if isinstance(d, dict) and d.get("value")]
    if isinstance(o, str):
        return re.findall(r"['\"]value['\"]\s*:\s*['\"]([^'\"]+)['\"]", o)
    return []

for c in ["abstract", "snippet", "section_name", "subsection_name", "news_desk", "kaggle_lead_paragraph"]:
    if c not in merged:
        merged[c] = ""
merged["headline"] = (merged["headline.main"] if "headline.main" in merged
                      else merged["headline"].map(headline_text) if "headline" in merged else "")
merged["keyword_list"] = merged["keywords"].map(keyword_list) if "keywords" in merged else [[]] * len(merged)
merged["keywords"] = merged["keyword_list"].map("; ".join)

def model_text(r):
    # headline + abstract + lead paragraph, skipping near-duplicates (abstract often == lead)
    parts, out = [r["headline"], r["abstract"] or r["snippet"], r["lead_paragraph"] or r["kaggle_lead_paragraph"]], []
    for p in parts:
        p = re.sub(r"\s+", " ", str(p)).strip()
        if p and not any(p[:60].lower() in q.lower() for q in out):
            out.append(p)
    return " ".join(out)[:1500]

merged["model_text"] = merged.apply(model_text, axis=1)

# Independent "silver" label from NYT's OWN editors (never shown to the model): section, URL path, or subject tags
HEALTH_TAGS = re.compile(
    r"health|medicine|medical|disease|mental|psycholog|psychiatr|depression|anxiety|suicide|loneliness|"
    r"hospital|doctors|nurses|patients|drugs \(pharmaceuticals\)|vaccin|cancer|obesity|diet|nutrition|"
    r"exercise|fitness|sleep|addiction|brain|dementia|alzheimer|coronavirus|covid|epidemic|pandemic", re.I)
url_section = merged["web_url"].str.extract(r"nytimes\.com/\d{4}/\d{2}/\d{2}/([^/]+)/", expand=False).fillna("").str.lower()
merged["nyt_tag_health"] = (
    merged["section_name"].str.lower().isin(["health", "well", "wellness"])
    | merged["news_desk"].str.lower().isin(["health", "well", "wellness"])
    | url_section.isin(["health", "well"])
    | merged["keyword_list"].map(lambda ks: any(HEALTH_TAGS.search(k) for k in ks))
).astype(int)

print(merged.loc[merged["group"] == "ai", ["headline", "section_name", "keywords"]].head(5).to_string())
print("\nExample model input:\n", merged["model_text"].iloc[0])

In [ ]:
# 2.7 · Zero-shot health score (about 2-4 minutes on a T4 GPU)
from transformers import pipeline

clf = pipeline("zero-shot-classification", model=MODEL_NAME, device=DEVICE)
label_text = list(HEALTH_LABELS.values())
label_key = {v: k for k, v in HEALTH_LABELS.items()}

texts = [t if t.strip() else "(no text)" for t in merged["model_text"].tolist()]
results, t0, STEP = [], time.time(), 128
for i in range(0, len(texts), STEP):
    out = clf(texts[i:i + STEP], candidate_labels=label_text, hypothesis_template=HYPOTHESIS_TEMPLATE,
              multi_label=True, batch_size=32)      # multi_label=True -> an independent 0-1 probability per label
    results += [out] if isinstance(out, dict) else out
    print(f"  scored {min(i + STEP, len(texts)):>5,}/{len(texts):,}  ({time.time() - t0:4.0f}s)")

sub = pd.DataFrame([{f"score_{label_key[l]}": s for l, s in zip(r["labels"], r["scores"])} for r in results],
                   index=merged.index)
merged = merged.drop(columns=[c for c in sub.columns if c in merged]).join(sub)
score_cols = [f"score_{k}" for k in HEALTH_LABELS]
merged["health_score"] = merged[score_cols].max(axis=1)
merged["health_topic"] = merged[score_cols].idxmax(axis=1).str.replace("score_", "", regex=False)
merged["model"] = MODEL_NAME
print(merged.groupby("group")["health_score"].describe().round(3))

In [ ]:
# 2.8 · Does the model agree with NYT's own editors? (this is your validation evidence)
from sklearn.metrics import roc_auc_score, precision_score, recall_score

y, s = merged["nyt_tag_health"], merged["health_score"]
pred = (s >= THRESHOLD).astype(int)
if y.nunique() == 2:
    print(f"ROC AUC vs NYT editorial health tags: {roc_auc_score(y, s):.3f}   (0.5 = coin flip, 1.0 = perfect)")
print(f"At threshold {THRESHOLD}: precision {precision_score(y, pred, zero_division=0):.2f} | "
      f"recall {recall_score(y, pred, zero_division=0):.2f} | "
      f"{pred.mean():.1%} of articles flagged as health")

pd.set_option("display.max_colwidth", 90)
ai = merged[merged["group"] == "ai"].sort_values("health_score", ascending=False)
print("\nMost health-related AI articles (read these: do they make sense?)")
print(ai[["health_score", "health_topic", "headline"]].head(12).to_string(index=False))
print("\nLeast health-related AI articles")
print(ai[["health_score", "headline"]].tail(6).to_string(index=False))
print("\nPossible misses: NYT tagged it as health, model scored < threshold")
print(merged.loc[(y == 1) & (pred == 0), ["group", "health_score", "headline", "keywords"]].head(6).to_string(index=False))

In [ ]:
# 2.9 · Save and download ai_health_scores.csv  -> upload it to DataHub next to hw03.ipynb
OUT_COLS = ["web_url", "group", "headline", "section_name", "news_desk", "keywords",
            "health_score", "health_topic", *score_cols, "nyt_tag_health", "model"]
merged[OUT_COLS].to_csv("ai_health_scores.csv", index=False)
print(f"Saved ai_health_scores.csv: {len(merged):,} rows")
files.download("ai_health_scores.csv")